# Dataset linting

Run quality checks on a dataset with the dataset linter API. List available rules, execute a lint run, inspect results (summary, per-rule stats, issues), and browse past runs for the same dataset.

In [1]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Choose a dataset

Set the `DATASET_ID` environment variable to lint an existing dataset. If unset, the notebook creates an empty dataset so the example still runs (replace with a dataset that has samples for meaningful lint output).

In [3]:
import os

dataset_id = os.environ.get("LIGHTNINGROD_DATASET_ID")
print("Using dataset:", dataset_id)

Using dataset: ad98afcb-da82-4f7b-ada5-5447122e992c


## List available linter rules

Each rule has a `name` and `default_severity`.

In [4]:
from pprint import pprint

rules_response = lr.datasets.linter.list_rules()
pprint([r.to_dict() for r in rules_response.rules])

[{'default_severity': 'warning', 'name': 'base_rate'},
 {'default_severity': 'warning', 'name': 'prediction_date_distribution'},
 {'default_severity': 'warning', 'name': 'forecast_horizon_distribution'},
 {'default_severity': 'warning', 'name': 'reward_distribution'},
 {'default_severity': 'warning', 'name': 'llm_contamination'},
 {'default_severity': 'warning', 'name': 'llm_context_relevance'},
 {'default_severity': 'warning', 'name': 'llm_question_comprehension'},
 {'default_severity': 'warning', 'name': 'llm_rollout_quality'}]


## Run the linter

Call `run()` with no extra arguments to use the server defaults (all rules, default sample size for LLM-backed rules). Pass `rules=[...]` and/or `sample_size=...` to override.

In [5]:
run_result = lr.datasets.linter.run(dataset_id)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Dataset Linter: COMPLETED                                                                                   │
│                                                                                                                 │
│    Run ID:      8b1bba33-77d1-4f1c-aaca-da28bd0c3332                                                            │
│    Dataset:     ad98afcb-da82-4f7b-ada5-5447122e992c                                                            │
│    Created:     2026-04-29T12:15:29.179000+00:00                                                                │
│    Updated:     2026-04-29T12:19:10.507000+00:00                                                                │
│    Sample size: 200                                                                                             │
│                                                                                                                 │
│    Issues:      62                                                                                              │
│    Severity:    error: 2, warning: 59, info: 1                                                                  │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓  │
│  ┃ Rule                                               ┃       Issues ┃ Severity             ┃       Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩  │
│  │ llm_context_relevance                              │           58 │ warning: 58          │          53.0s │  │
│  │ forecast_horizon_distribution                      │            1 │ error: 1             │           27ms │  │
│  │ llm_contamination                                  │            1 │ warning: 1           │         100.0s │  │
│  │ prediction_date_distribution                       │            1 │ error: 1             │           32ms │  │
│  │ reward_distribution                                │            1 │ info: 1              │            9ms │  │
│  │ base_rate                                          │            0 │ 0                    │          107ms │  │
│  │ llm_question_comprehension                         │            0 │ 0                    │          64.2s │  │
│  │ llm_rollout_quality                                │            0 │ 0                    │           44ms │  │
│  └────────────────────────────────────────────────────┴──────────────┴──────────────────────┴────────────────┘  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [6]:
from lightningrod import display_lint_detailed

display_lint_detailed(run_result)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Dataset Linter Details                                                                                      │
│                                                                                                                 │
│    Run ID:  8b1bba33-77d1-4f1c-aaca-da28bd0c3332                                                                │
│    Dataset: ad98afcb-da82-4f7b-ada5-5447122e992c                                                                │
│                                                                                                                 │
│  forecast_horizon_distribution (27ms, 1 issue)                                                                  │
│   cv                                       1.0115739219561188                                                   │
│   max_days                                 548.0                                                                │
│   mean_days                                96.43511450381679                                                    │
│   min_days                                 1.0                                                                  │
│   p50_days                                 65.0                                                                 │
│   p95_days                                 279.4                                                                │
│   std_days                                 97.55124699291335                                                    │
│   valid_rows                               435                                                                  │
│                                                                                                                 │
│    Issue 1: error                                                                                               │
│      Message: 42 row(s) have non-positive forecast horizon.                                                     │
│      Affected samples: 41f79a84-be7c-4e8a-9689-fd97a9ce4618                                                     │
│  44ce29d7-5aeb-449a-98d2-ea95f93b4857                                                                           │
│  6654be85-019e-4ec2-811a-e709e9d49b92                                                                           │
│  83096262-9abe-4206-89ba-57b2106cb3c0                                                                           │
│  4f71817f-3a6c-4b25-b255-6aaa405dc187                                                                           │
│  cfd58473-0783-4b95-942d-863e06e80376                                                                           │
│  3133969c-c570-4bdc-910a-373a8c5223f8                                                                           │
│  5ed8ef00-20e8-45a1-80cb-680e6d810ee7                                                                           │
│  04644d1b-6c3d-4f89-bc04-6063a3d9d503                                                                           │
│  c2dded50-3409-4ff7-8590-6d69efe57ea4                                                                           │
│  +32 more                                                                                                       │
│      Meta: {"count": 42}                                                                                        │
│      Tip: The resolution must occur strictly after the prediction date.                                         │
│                                                                                                                 │
│  prediction_date_distribution (32ms, 1 issue)                                                                   │
│   clump_share                         0.16091954022988

## Compile samples to review or exclude

Use `get_lint_affected_sample_ids` to get the unique sample IDs attached to warning/error lint issues. These are candidates to review before creating train/test splits.

In [7]:
from lightningrod import get_lint_affected_sample_ids

affected_sample_ids = get_lint_affected_sample_ids(run_result)
print(f"{len(affected_sample_ids)} sample(s) affected by warning/error lint issues")

dataset = lr.datasets.get(dataset_id)
filtered_dataset = dataset.exclude(affected_sample_ids)
print(f"Kept {filtered_dataset.num_rows} of {dataset.num_rows} sample(s)")
# Use the filtered dataset for training, testing, etc.

101 sample(s) affected by warning/error lint issues
Kept 334 of 435 sample(s)


## Optional: run a subset of rules

Uncomment and set rule names from the list above.

In [8]:
from lightningrod import display_lint_detailed

rule_names = [r.name for r in rules_response.rules[:2]]
subset_run = lr.datasets.linter.run(dataset_id, rules=rule_names, random_sample_size=50)
display_lint_detailed(subset_run)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Dataset Linter: COMPLETED                                                                                   │
│                                                                                                                 │
│    Run ID:      be1f533b-7991-4ec9-a260-e4ae09f1ee09                                                            │
│    Dataset:     ad98afcb-da82-4f7b-ada5-5447122e992c                                                            │
│    Created:     2026-04-29T12:19:36.615000+00:00                                                                │
│    Updated:     2026-04-29T12:19:38.349000+00:00                                                                │
│    Sample size: 50                                                                                              │
│                                                                                                                 │
│    Issues:      1                                                                                               │
│    Severity:    error: 1                                                                                        │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓  │
│  ┃ Rule                                                 ┃        Issues ┃ Severity         ┃        Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩  │
│  │ prediction_date_distribution                         │             1 │ error: 1         │            26ms │  │
│  │ base_rate                                            │             0 │ 0                │            68ms │  │
│  └──────────────────────────────────────────────────────┴───────────────┴──────────────────┴─────────────────┘  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Dataset Linter Details                                                                                      │
│                                                                                                                 │
│    Run ID:  be1f533b-7991-4ec9-a260-e4ae09f1ee09                                                                │
│    Dataset: ad98afcb-da82-4f7b-ada5-5447122e992c                                                                │
│                                                                                                                 │
│  prediction_date_distribution (26ms, 1 issue)                                                                   │
│   clump_share                         0.16091954022988506                                                       │
│   max_date                            2025-12-27T00:00:00+00:00                                                 │
│   min_date                            2024-07-30T00:00:00+00:00                                                 │
│   span_days                           515                                                                       │
│   unique_days                         46                                                                        │
│   valid_rows                          435                                                                       │
│                                                                                                                 │
│    Issue 1: error                                                                                               │
│      Message: 42 row(s) have prediction_date >= resolution_date.                                                │
│      Affected samples: 41f79a84-be7c-4e8a-9689-fd97a9ce4618                                                     │
│  44ce29d7-5aeb-449a-98d2-ea95f93b4857                                                                           │
│  6654be85-019e-4ec2-811a-e709e9d49b92                                                                           │
│  83096262-9abe-4206-89ba-57b2106cb3c0                                                                           │
│  4f71817f-3a6c-4b25-b255-6aaa405dc187                                                                           │
│  cfd58473-0783-4b95-942d-863e06e80376                                                                           │
│  3133969c-c570-4bdc-910a-373a8c5223f8                                                                           │
│  5ed8ef00-20e8-45a1-80cb-680e6d810ee7                                                                           │
│  04644d1b-6c3d-4f89-bc04-6063a3d9d503                                                                           │
│  c2dded50-3409-4ff7-8590-6d69efe57ea4                                                                           │
│  +32 more                                                                                                       │
│      Meta: {"count": 42}                                                                                        │
│      Tip: Temporal leakage: the model would see information from after it needs to predict.                     │
│                                                                                                                 │
│  base_rate (68ms, 0 issues)                                                                                     │
│   class_counts                           {"0.0": 246, "1.0": 141}                                               │
│   max_share                              0.6356589147286822                                                     │
│   num_classes                            2            

## Fetch a run by id and list past runs

`get_run` returns the same shape as `run`. `list_runs` returns a lightweight list of runs for the dataset.

In [9]:
by_id = lr.datasets.linter.get_run(run_result.id)
assert by_id.id == run_result.id

In [10]:
past = lr.datasets.linter.list_runs(dataset_id, limit=10)
pprint([r.to_dict() for r in past.runs])

[{'created_at': '2026-04-29T12:19:36.615000+00:00',
  'dataset_id': 'ad98afcb-da82-4f7b-ada5-5447122e992c',
  'error_message': None,
  'id': 'be1f533b-7991-4ec9-a260-e4ae09f1ee09',
  'status': 'COMPLETED',
  'total_issues': 1,
  'updated_at': '2026-04-29T12:19:38.349000+00:00'},
 {'created_at': '2026-04-29T12:15:29.179000+00:00',
  'dataset_id': 'ad98afcb-da82-4f7b-ada5-5447122e992c',
  'error_message': None,
  'id': '8b1bba33-77d1-4f1c-aaca-da28bd0c3332',
  'status': 'COMPLETED',
  'total_issues': 62,
  'updated_at': '2026-04-29T12:19:10.507000+00:00'},
 {'created_at': '2026-04-29T09:59:23.306000+00:00',
  'dataset_id': 'ad98afcb-da82-4f7b-ada5-5447122e992c',
  'error_message': None,
  'id': '9ee24cbb-5fa7-4c2a-9c28-e7c4662d1ecc',
  'status': 'COMPLETED',
  'total_issues': 1,
  'updated_at': '2026-04-29T09:59:25.504000+00:00'},
 {'created_at': '2026-04-29T09:56:07.613000+00:00',
  'dataset_id': 'ad98afcb-da82-4f7b-ada5-5447122e992c',
  'error_message': None,
  'id': 'a93df56c-a55c-4e3